In [8]:
import torch
import tqdm

class IoU:
    def __init__(self, num_classes, ignore_index=None, device="cpu"):
        self.num_classes = num_classes
        self.device = device
        self.conf_matrix = torch.zeros((num_classes, num_classes),
                                       dtype=torch.int64,
                                       device=device)

        if ignore_index is None:
            self.ignore_index = None
        elif isinstance(ignore_index, int):
            self.ignore_index = {ignore_index}
        else:
            self.ignore_index = set(ignore_index)

    def reset(self):
        self.conf_matrix.zero_()

    def _fast_hist(self, pred, target):
        pred = pred.long()
        target = target.long()

        mask = (target >= 0) & (target < self.num_classes)

        if self.ignore_index is not None:
            for idx in self.ignore_index:
                mask = mask & (target != idx)
        pred = pred[mask]
        target = target[mask]

        hist = torch.bincount(
            self.num_classes * target + pred,
            minlength=self.num_classes * self.num_classes
        ).reshape(self.num_classes, self.num_classes)

        return hist.to(self.device)

    def add(self, predicted, target):
        if predicted.dim() == 4:
            predicted = torch.argmax(predicted, dim=1)
        if target.dim() == 4:
            target = torch.argmax(target, dim=1)

        predicted = predicted.view(-1).to(self.device)
        target = target.view(-1).to(self.device)

        self.conf_matrix += self._fast_hist(predicted, target)

    def value(self):
        conf = self.conf_matrix.float().clone()

        if self.ignore_index is not None:
            for idx in self.ignore_index:
                conf[idx, :] = 0
                conf[:, idx] = 0

        tp = torch.diag(conf)
        fp = conf.sum(dim=0) - tp
        fn = conf.sum(dim=1) - tp

        iou = tp / (tp + fp + fn + 1e-10)
        miou = torch.nanmean(iou)

        return iou, miou

def compute_iou(model, loader, num_classes, device="cuda", ignore_index=None):
    model.eval()

    iou_metric = IoU(num_classes=num_classes,
                     ignore_index=ignore_index,
                     device=device)

    iou_metric.reset()

    with torch.no_grad():
        for item in tqdm(loader, desc="Computing IoU"):
            img  = item["image"].to(device)
            mask = item["classification"].to(device)

            pred = model(img)[0]   # (N,C,H,W)

            iou_metric.add(pred, mask)

    del img, mask, pred
    return iou_metric.value()

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from model import FullModelHAT
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from dataset import *
from torch.utils.data import WeightedRandomSampler
from tqdm import tqdm

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: ", device)

# ---------------- CONFIG ----------------
version = "V13"

checkpoint_path = f"rescuenet_seg_{version}.pth"
checkpoint  = checkpoint_path if os.path.exists(checkpoint_path) else None

num_classes = 11
batch_size  = 24
workers     = 8
epochs      = 500
lr          = 3e-4
boundary_weight = 0.3

root = "./rescuenet"
# image_size = (256, 256)
image_size = (384, 384)

# ---------------- DATA ----------------
train_dataset = RescueNetDataset(root=root, split='train', image_size=image_size, augment=True)
val_dataset   = RescueNetDataset(root=root, split='val',   image_size=image_size, augment=False)


# training_weights = compute_sample_weights_fast(train_dataset, "cuda")
if f"training_weights{image_size[0]}.npy" in os.listdir():
    training_weights = np.load(f"training_weights{image_size[0]}.npy")
    print("Loaded Training weights")
else:
    training_weights = compute_sample_weights_fast(train_dataset, device)
    np.save(f"training_weights{image_size[0]}.npy", training_weights)

sampler = WeightedRandomSampler(
    weights=training_weights,
    num_samples=len(training_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, 
                          batch_size=batch_size,
                          sampler=sampler,
                        #   shuffle=True,
                        #   num_workers=workers, 
                        #   pin_memory=True, 
                        #   persistent_workers=True
                          )

val_loader = DataLoader(val_dataset, 
                        batch_size=batch_size, 
                        shuffle=False,
                        # num_workers=workers, 
                        # pin_memory=True, 
                        # persistent_workers=True
                        )

# ---------------- COLORS ----------------
CLASS_COLORS = np.array([
    [0,0,0],[61,230,250],[180,120,120],[235,255,7],[255,184,6],
    [255,0,0],[255,0,245],[140,140,140],[160,150,20],[4,250,7],[255,235,0]
], dtype=np.uint8)

def mask_to_color(mask):
    return CLASS_COLORS[mask]

# ---------------- LOSS ----------------
# weights = torch.tensor([
#     0.0972,0.4650,0.9944,0.9752,1.2114,
#     1.2534,1.8995,0.5233,1.2445,0.2080,2.1282
# ], dtype=torch.float32).to(device)

weights = torch.tensor([
    0.1, 0.5,
    1.0,
    1.2,
    1.5,
    1.8,
    1.5,
    0.5,
    2.2,   # reduce from 3.0
    0.3,
    1.8    # reduce from 2.5
]).to(device)

Device:  cpu
[RescueNetDataset] split=train | samples=3595 | augment=True
[RescueNetDataset] split=val   | samples=449 | augment=False
Loaded Training weights


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, ignore_index=255):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.ignore_index = ignore_index

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.weight,
                             reduction='none', ignore_index=self.ignore_index)
        pt = torch.exp(-ce).clamp(1e-6, 1.0)
        loss = (1 - pt) ** self.gamma * ce

        valid = targets != self.ignore_index
        return loss[valid].mean()


def lovasz_grad(gt_sorted):
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def lovasz_softmax_flat(probs, labels, classes='present'):
    C = probs.shape[1]
    losses = []
    present_classes = torch.unique(labels)

    for c in range(C):
        if classes == 'present' and c not in present_classes:
            continue
        fg = (labels == c).float()
        if fg.sum() == 0:
            continue
        errors = (fg - probs[:, c]).abs()
        errors_sorted, perm = torch.sort(errors, descending=True)
        fg_sorted = fg[perm]
        losses.append(torch.dot(errors_sorted, lovasz_grad(fg_sorted)))

    if len(losses) == 0:
        return probs.sum() * 0.0

    return torch.stack(losses).mean()


class LovaszSoftmax(nn.Module):
    def __init__(self, ignore_index=255, classes='present'):
        super().__init__()
        self.ignore_index = ignore_index
        self.classes = classes

    def forward(self, inputs, targets):
        probs = F.softmax(inputs, dim=1)

        B, C, H, W = probs.shape
        probs   = probs.permute(0, 2, 3, 1).reshape(-1, C)  # (B*H*W, C)
        targets = targets.reshape(-1)                         # (B*H*W,)

        valid   = targets != self.ignore_index
        probs   = probs[valid]
        targets = targets[valid]

        return lovasz_softmax_flat(probs, targets, classes=self.classes)


class CombinedLoss(nn.Module):
    def __init__(self, class_weights=None, gamma=2.0, ignore_index=255,
                 focal_w=0.5, lovasz_w=0.5):
        super().__init__()
        self.focal   = FocalLoss(weight=class_weights, gamma=gamma,
                                 ignore_index=ignore_index)
        self.lovasz  = LovaszSoftmax(ignore_index=ignore_index, classes='present')
        self.focal_w  = focal_w
        self.lovasz_w = lovasz_w

    def forward(self, inputs, targets):
        return self.focal_w  * self.focal(inputs, targets) + self.lovasz_w * self.lovasz(inputs, targets)

seg_loss = CombinedLoss(
    class_weights=weights,
    gamma=2.0,
    ignore_index=255,
    focal_w=0.4,
    lovasz_w=0.6
)

boundary_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([15.0]).to(device))

# ---------------- IoU ----------------
def compute_miou(model, loader):
    model.eval()
    conf = torch.zeros((num_classes,num_classes), device=device)

    with torch.inference_mode():
        for item in tqdm(loader, leave=False):
            img = item["image"].to(device)
            mask = item["classification"].to(device)

            if mask.ndim==4: mask = mask.squeeze(1)

            outputs = model(img)
            pred = outputs[0]
            pred = torch.argmax(pred, dim=1)

            pred = pred.view(-1)
            target = mask.view(-1)

            k = (target>=0)&(target<num_classes)
            inds = num_classes*target[k]+pred[k]

            conf += torch.bincount(inds, minlength=num_classes**2).reshape(num_classes,num_classes)

    tp = torch.diag(conf)
    fp = conf.sum(0)-tp
    fn = conf.sum(1)-tp

    iou = tp/(tp+fp+fn+1e-6)
    present = (tp+fp+fn)>0
    return iou[present].mean().item()

# ---------------- VALIDATE ----------------
def validate(model, loader):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for item in tqdm(loader, leave=False, desc="Validating: "):
            img  = item["image"].to(device)
            mask = item["classification"].to(device)
            edge = item["boundary"].to(device)

            if mask.ndim == 4:
                mask = mask.squeeze(1)
            if edge.ndim == 3:
                edge = edge.unsqueeze(1)

            edge = edge.float().clamp(0,1)

            outputs = model(img)

            if len(outputs) == 3:
                pred_mask, _, pred_edge = outputs
            else:
                pred_mask, pred_edge = outputs

            seg_l = seg_loss(pred_mask, mask)

            edge_t = F.interpolate(edge, size=pred_edge.shape[-2:], mode='nearest')
            boundary_l = boundary_criterion(pred_edge, edge_t)

            loss = seg_l + boundary_weight * boundary_l
            total_loss += loss.item()
        del img, mask, edge, outputs
    return total_loss / len(loader)

def save_validation_examples(model, epoch, val_loader, device, save_dir="outputs", num_samples=3):
    os.makedirs(save_dir, exist_ok=True)
    model.eval()
    images = []

    with torch.no_grad():
        for item in val_loader:
            img  = item["image"].to(device)
            mask = item["classification"].to(device)
            edge = item["boundary"].to(device)

            outputs = model(img)
            if len(outputs)==3:
                    pred_mask, _, pred_edge = outputs
            else:
                pred_mask, pred_edge = outputs

            pred_mask = torch.argmax(pred_mask, dim=1)

            # pred_edge is raw logits — apply sigmoid here for visualisation only
            pred_edge = (torch.sigmoid(pred_edge) > 0.7).float()
            # pred_edge = torch.sigmoid(pred_edge)

            for b in range(img.shape[0]):
                mean = torch.tensor([0.485, 0.456, 0.406], device=img.device).view(3,1,1)
                std  = torch.tensor([0.229, 0.224, 0.225], device=img.device).view(3,1,1)

                img_denorm = img[b] * std + mean

                image_np = img_denorm.permute(1, 2, 0).cpu().numpy()
                image_np = np.clip(image_np, 0, 1)
                image_vis = (image_np * 255).astype(np.uint8)

                gt_mask_raw = mask[b].cpu().numpy()
                pr_mask_raw = pred_mask[b].cpu().numpy()

                gt_mask  = mask_to_color(gt_mask_raw)
                pr_mask  = mask_to_color(pr_mask_raw)
                # image_vis = (image_np * 255).astype(np.uint8) if image_np.max() <= 1 else image_np

                gt_overlay = (0.6 * image_vis + 0.4 * gt_mask).astype(np.uint8)
                pr_overlay = (0.6 * image_vis + 0.4 * pr_mask).astype(np.uint8)

                gt_edge = edge[b].squeeze().cpu().numpy()
                pr_edge = pred_edge[b].squeeze().cpu().numpy()

                images.append((image_vis, gt_overlay, gt_edge, pr_overlay, pr_edge))

            if len(images) >= num_samples:
                break

    fig, axs = plt.subplots(num_samples, 5, figsize=(20, 5 * num_samples))
    titles = ["Image", "GT Mask", "GT Boundary", "Pred Mask", "Pred Boundary"]

    for i in range(num_samples):
        for j in range(5):
            if j in [2, 4]:
                axs[i, j].imshow(images[i][j], cmap='gray')
            else:
                axs[i, j].imshow(images[i][j])
            axs[i, j].set_title(titles[j])
            axs[i, j].axis('off')

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"combined_samples{epoch}.png")
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()

# ---------------- TRAIN ----------------
def train():

    model = FullModelHAT(num_classes=num_classes).to(device)
    model = model.to(memory_format=torch.channels_last)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-7
    )


    scaler = torch.cuda.amp.GradScaler()
    best_val = 0.0
    start_epoch = 0

    if checkpoint is not None:
        ckpt = torch.load(checkpoint, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optim_state'])
        scheduler.load_state_dict(ckpt['sched_state'])   # was commented out before
        best_val    = ckpt.get('best_val', 0.0)
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed from epoch {start_epoch}, best val loss: {best_val:.4f}")


    print(f"Training on {device} | epochs {start_epoch}→{epochs} | classes {num_classes} | Loaded {checkpoint}")

    for epoch in range(start_epoch, epochs):
        model.train()
        total_loss = 0

        for batch_idx, item in enumerate(tqdm(train_loader, leave=False, desc="Training")):
            img  = item["image"].to(device, memory_format=torch.channels_last)
            mask = item["classification"].to(device)
            edge = item["boundary"].to(device)

            if mask.ndim==4: mask=mask.squeeze(1)
            if edge.ndim==3: edge=edge.unsqueeze(1)
            edge = edge.float().clamp(0,1)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast():

                outputs = model(img)

                if len(outputs)==3:
                    pred_mask, aux_mask, pred_edge = outputs
                else:
                    pred_mask, pred_edge = outputs
                    aux_mask=None

                loss_main = seg_loss(pred_mask,mask)
                loss_aux  = seg_loss(aux_mask, mask) if aux_mask is not None else 0

                edge_t = F.interpolate(edge, size=pred_edge.shape[-2:], mode='nearest')
                loss_edge = boundary_criterion(pred_edge,edge_t)

                loss = loss_main + 0.5*loss_aux + boundary_weight*loss_edge

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()

            total_loss+=loss.item()

        with torch.no_grad():
            val_loss = validate(model, val_loader)
            _, miou = compute_iou(
                model,
                val_loader,
                num_classes=num_classes,
                device=device,
                ignore_index=0
            )
            scheduler.step()

            del img, mask, outputs
            torch.cuda.empty_cache()

        current_lr = optimizer.param_groups[0]['lr']


        txt = f"Epoch {epoch} | Train {total_loss/len(train_loader):.4f} | Val {val_loss:.4f} | LR {current_lr:.7f} | mIoU {miou:.4f}"

        print(txt)

        log_file = f"training_log_v{version}.txt"
        with open(log_file, "a") as f:
            f.write(txt + "\n")

        if epoch % 5 == 0:
            save_validation_examples(
                model, epoch, val_loader, device,
                save_dir=f"outputs/{version}", num_samples=5,
            )


        if miou > best_val:
            best_val = miou
            torch.save({
                'epoch':       epoch,
                'model_state': model.state_dict(),
                'optim_state': optimizer.state_dict(),
                'sched_state': scheduler.state_dict(),
                'best_val':    best_val,
            }, checkpoint_path)
            print(f"-> Saved best model (IoU: {best_val:.4f})")

        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

if __name__ == "__main__":
    train()
    # pass

[RescueNetDataset] split=train | samples=3595 | augment=True
[RescueNetDataset] split=val   | samples=449 | augment=False
Loaded Training weights


C:\Users\Jaitej\AppData\Local\Temp\ipykernel_44448\845558115.py:338: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
c:\Users\Jaitej\anaconda3\envs\dlcv-1\Lib\site-packages\torch\cuda\amp\grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


Resumed from epoch 498, best val loss: 0.7118
Training on cpu | epochs 498→500 | classes 11 | Loaded rescuenet_seg_V13.pth


Training:   0%|          | 0/150 [00:00<?, ?it/s]c:\Users\Jaitej\anaconda3\envs\dlcv-1\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\Jaitej\AppData\Local\Temp\ipykernel_44448\845558115.py:369: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
c:\Users\Jaitej\anaconda3\envs\dlcv-1\Lib\site-packages\torch\cuda\amp\autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(


KeyboardInterrupt: 

In [10]:
model = FullModelHAT(num_classes=11).to(device)

ignore_index=0
checkpoint = "rescuenet_seg_V13.pth"
ckpt = torch.load(checkpoint, map_location=device)
model.load_state_dict(ckpt['model_state'])
print("Loaded: ", checkpoint)


test_dataset   = RescueNetDataset(root=root, split='test',   image_size=image_size, augment=False)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    # num_workers=4,
    # pin_memory=True,
    # persistent_workers=True,
)

iou_per_class, miou = compute_iou(
    model,
    test_loader,
    num_classes=11,
    device=device,
    ignore_index=ignore_index
)

CLASS_NAMES = [
    'Background',
    'Water',
    'Building_No_Damage',
    'Building_Minor_Damage',
    'Building_Major_Damage',
    'Building_Total_Destruction',
    'Vehicle',
    'Road-Clear',
    'Road-Blocked',
    'Tree',
    'Pool'
]

print(f"\n{'Class':<30} {'IoU':>8} {'IoU %':>8}")
print("-" * 50)

count=0
s = 0
for i, name in enumerate(CLASS_NAMES):
    if i == ignore_index:
        continue
    val = iou_per_class[i].item()
    count += 1
    s += val
    print(f"{name:<30} {val:>8.4f} {val*100:>7.2f}%")

print("-" * 50)
miou = s/count
print(f"{'Mean IoU':<30} {miou:>8.4f} {miou*100:>7.2f}%")

Loaded:  rescuenet_seg_V13.pth
[RescueNetDataset] split=test  | samples=450 | augment=False


Computing IoU: 100%|██████████| 225/225 [09:03<00:00,  2.42s/it]


Class                               IoU    IoU %
--------------------------------------------------
Water                            0.0341    3.41%
Building_No_Damage               0.0007    0.07%
Building_Minor_Damage            0.0820    8.20%
Building_Major_Damage            0.0843    8.43%
Building_Total_Destruction       0.0104    1.04%
Vehicle                          0.0000    0.00%
Road-Clear                       0.0036    0.36%
Road-Blocked                     0.1661   16.61%
Tree                             0.8560   85.60%
Pool                             0.7230   72.30%
--------------------------------------------------
Mean IoU                         0.1960   19.60%
